# The Quantum Prisoner's Dilemma — Eisert–Wilkens–Lewenstein, in Qiskit

**Eleanor Adams** · companion to my paper *“How can quantum algorithms provide insight into economic outcomes?”*

This notebook reproduces the interactive simulator at the repo's homepage on real quantum tooling.
We implement the full **EWL protocol** — a tunable entangling gate $\hat J(\gamma)$, the two-parameter
strategy space $\hat U(\theta,\varphi)$, and the classical/quantum strategies $C, D, Q$ — then show the
headline result: with enough entanglement, **mutual cooperation $(Q,Q)$ becomes a Nash equilibrium** and
the dilemma disappears — passing through **two** exact thresholds, $\sin^2\gamma=1/5$ and $2/5$.


## 1 · Setup

In [ ]:
import sys
!{sys.executable} -m pip install -q 'qiskit>=1.0' 'qiskit-aer>=0.14' numpy matplotlib pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.circuit.library import UnitaryGate, RYYGate

# Payoff matrix (Alice, Bob), indexed by measured bitstring b0b1 with 0=Cooperate, 1=Defect.
#   CC=(3,3)  CD=(0,5)  DC=(5,0)  DD=(1,1)
PAYOFF = {'00': (3, 3), '01': (0, 5), '10': (5, 0), '11': (1, 1)}

def U(theta, phi):
    """EWL 2-parameter strategy: a unitary rotation of one player's qubit."""
    return np.array([[np.exp(1j*phi)*np.cos(theta/2),  np.sin(theta/2)],
                     [-np.sin(theta/2),                np.exp(-1j*phi)*np.cos(theta/2)]], dtype=complex)

# The three named strategies
C = U(0, 0)          # Cooperate  = identity
D = U(np.pi, 0)      # Defect     = bit flip (iY)
Q = U(0, np.pi/2)    # Quantum    = iZ  (no classical counterpart)

## 2 · The EWL circuit

The referee entangles the two qubits with $\hat J(\gamma)=\exp\!\big(i\tfrac{\gamma}{2}\,\sigma_y\otimes\sigma_y\big)$.
Qiskit's `RYYGate(\lambda)` is $\exp(-i\tfrac{\lambda}{2}\,\sigma_y\otimes\sigma_y)$, so $\hat J(\gamma)=$`RYY(-\gamma)`
and $\hat J^\dagger(\gamma)=$`RYY(\gamma)`. Each player then applies their private strategy; the referee
un-entangles and measures.

In [ ]:
def ewl_circuit(UA, UB, gamma, measure=True):
    qc = QuantumCircuit(2, 2)
    qc.append(RYYGate(-gamma), [0, 1])          # J(gamma): entangle
    qc.barrier()
    qc.append(UnitaryGate(UA, label='U_A'), [0])  # Alice's strategy
    qc.append(UnitaryGate(UB, label='U_B'), [1])  # Bob's strategy
    qc.barrier()
    qc.append(RYYGate(gamma), [0, 1])           # J^dagger: un-entangle
    if measure:
        qc.measure([0, 1], [0, 1])
    return qc

print('Full EWL circuit — both players choose Q, maximal entanglement:')
ewl_circuit(Q, Q, np.pi/2).draw('mpl')

## 3 · Exact payoffs (state-vector)

Because it's only two qubits, we can read the outcome probabilities exactly from the state vector
instead of sampling.

In [ ]:
def exact_payoffs(UA, UB, gamma):
    qc = ewl_circuit(UA, UB, gamma, measure=False)
    probs = Statevector(qc).probabilities_dict()   # keys are 'b1b0' (Qiskit little-endian)
    a = b = 0.0
    for bits, pr in probs.items():
        key = bits[::-1]        # reverse to 'b0b1' = (Alice, Bob)
        pa, pb = PAYOFF[key]
        a += pa * pr; b += pb * pr
    return a, b

g = np.pi/2
print(f'{"Alice vs Bob":16} {"payoff (A, B)"}')
print('-'*34)
for name, (ua, ub) in {'C vs C': (C, C), 'D vs D': (D, D), 'Q vs Q': (Q, Q),
                       'Q vs D (miracle)': (Q, D), 'D vs Q': (D, Q)}.items():
    a, b = exact_payoffs(ua, ub, g)
    print(f'{name:16} ({a:.2f}, {b:.2f})')

You should see **Q vs Q → (3, 3)** (the cooperative payoff) while **D vs D → (1, 1)**, and the *miracle move*
**Q vs D → (5, 0)**: against a classical defector, the quantum player walks away with the maximum. That last
line is why a defector can't hold: Q punishes them.

## 4 · Sampling check on the Aer simulator

Same circuit, but sampled with shots — this is what would run on hardware.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

sim = AerSimulator()
qc = ewl_circuit(Q, Q, np.pi/2)
counts = sim.run(transpile(qc, sim), shots=4096).result().get_counts()
print('Counts (Q, Q):', counts)
plt.bar([k[::-1] for k in counts], counts.values(), color='#a78bfa')
plt.title('(Q, Q) at maximal entanglement — always CC'); plt.ylabel('shots'); plt.xlabel('outcome (Alice,Bob)')
plt.show()

## 5 · Where the dilemma breaks — two thresholds, not one

It is tempting to expect a single magic value of entanglement. There isn't one. As $\gamma$ increases the
game crosses **two** exact thresholds:

- **$\gamma_1=\arcsin\sqrt{1/5}\approx0.464$** — below it, mutual defection $(D,D)$ is still the Nash
  equilibrium (the classical dilemma survives).
- **$\gamma_2=\arcsin\sqrt{2/5}\approx0.685$** — above it, mutual cooperation $(Q,Q)$ becomes the Nash
  equilibrium and pays $(3,3)$.

Between them lies a **frustrated region with no pure-strategy equilibrium**. We locate both thresholds by
finding, for each $\gamma$, the best a deviator can do against a fixed $D$ or $Q$ opponent.


In [ ]:
def best_exploit(target, gamma, nt=48, nph=36):
    """Best payoff a deviating player can get against a fixed opponent strategy."""
    best = -1.0
    for t in np.linspace(0, np.pi, nt):
        for ph in np.linspace(0, np.pi/2, nph):
            best = max(best, exact_payoffs(U(t, ph), target, gamma)[0])
    return best

g1, g2 = np.arcsin(np.sqrt(1/5)), np.arcsin(np.sqrt(2/5))
gammas = np.linspace(0, np.pi/2, 60)
exploit_vs_Q = [best_exploit(Q, g) for g in gammas]   # drops to 3 at g2  -> (Q,Q) becomes NE
exploit_vs_D = [best_exploit(D, g) for g in gammas]   # rises above 1 at g1 -> (D,D) stops being NE

plt.figure(figsize=(7.5, 4.6))
plt.axvspan(0, g1, color='#b0392a', alpha=0.08, label='classical: (D,D) is NE')
plt.axvspan(g1, g2, color='#9c7611', alpha=0.12, label='frustrated: no pure NE')
plt.axvspan(g2, np.pi/2, color='#2f7d4f', alpha=0.08, label='quantum: (Q,Q) is NE')
plt.axhline(3, color='k', lw=0.8, ls=':')
plt.plot(gammas, exploit_vs_Q, color='#5b3bd6', lw=2.4, label='best exploit vs Q')
plt.plot(gammas, exploit_vs_D, color='#b0392a', lw=2.4, label='best exploit vs D')
for g, name in [(g1, r'$\gamma_1$'), (g2, r'$\gamma_2$')]:
    plt.axvline(g, color='#9c7611', ls='--', lw=1); plt.text(g+0.01, 4.7, name, color='#9c7611')
plt.xlabel(r'entanglement  $\gamma$'); plt.ylabel('Alice payoff'); plt.ylim(0, 5.3)
plt.title('Two tipping points: sin$^2\\gamma$ = 1/5 and 2/5'); plt.legend(fontsize=8, loc='center right')
plt.tight_layout(); plt.show()

print(f'gamma_1 = {g1:.4f} rad  (sin^2 = {np.sin(g1)**2:.3f})')
print(f'gamma_2 = {g2:.4f} rad  (sin^2 = {np.sin(g2)**2:.3f})')

## 6 · Alice's payoff landscape (Bob plays Q)

Her whole strategy space at maximal entanglement — the bright peak sits exactly on $Q$.

In [ ]:
nt, nph = 60, 45
thetas = np.linspace(0, np.pi, nt); phis = np.linspace(0, np.pi/2, nph)
land = np.array([[exact_payoffs(U(t, ph), Q, np.pi/2)[0] for t in thetas] for ph in phis])

plt.figure(figsize=(7,4.5))
im = plt.imshow(land, origin='lower', aspect='auto', cmap='RdYlGn', vmin=0, vmax=5,
                extent=[0, np.pi, 0, np.pi/2])
plt.colorbar(im, label='Alice payoff')
plt.scatter([0],[np.pi/2], marker='D', s=90, edgecolor='white', facecolor='none', lw=2, label='Q')
plt.xlabel('$\\theta$  (cooperate $\\to$ defect)'); plt.ylabel('$\\varphi$  (quantum phase)')
plt.title('Alice best-responds with Q'); plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

## 7 · Why this matters for economics

The classical dilemma's only Nash equilibrium is mutual defection $(1,1)$ — individual rationality producing a
collectively bad outcome. Entanglement doesn't change the payoff matrix or let the players communicate; it
changes the **correlation structure** they share before choosing. Past a high-enough entanglement threshold, that's enough to make cooperation the rational choice.

As I argue in my paper, this is the interesting claim for economics: the equilibria of strategic interactions —
oligopoly pricing, public-goods problems, international agreements — aren't fixed. They depend on what the
players are allowed to share. Wang (2022) carries the same machinery to Cournot duopoly, where a regulator
tuning $\gamma$ can keep a second firm alive and prevent a monopoly.

**Next steps to explore:** sweep the *joint* $(\theta_A,\varphi_A,\theta_B,\varphi_B)$ space for equilibria at
intermediate $\gamma$; add decoherence/noise to see how fragile the quantum equilibrium is; or extend to the
3-player GHZ game from Wang §6.1.